## Feature Creation Notebook

Reads cleaned_master_df.csv — one row per order, output of cleaning_dataset.ipynb.

**Pipeline:**
1. Load and re-parse datetimes
2. Safe features — pure arithmetic, no cross-row aggregation (Sections 2-4)
3. Train/test split — stratified on review_score
4. Fold-aware features — fit on train only, map to test/val (Section 5)
5. Composite motif indices — Frustration / Financial Burden / Consistency (Section 6)
6. Assemble X_train, X_test, save arrays (Sections 7-8)

**Leakage rule:** anything that aggregates across rows (seller stats, category stats, customer stats, zip-code norms) lives in Section 5 and must be refit inside every CV fold using only that fold's training data.

**Target: 100 features total.**

Note: seller_id and customer_unique_id are kept in the dataframe through to prepare_X, which drops them there. They are needed as groupby keys in the fold-aware feature block.


### 1. Load cleaned dataset

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('../Data/cleaned_master_df.csv', index_col=0)

# Re-parse datetimes (CSV round-trips lose dtype)
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'shipping_limit_date',
    'review_answer_timestamp',
    'review_creation_date'
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Cast review_score to int for stratification
df['review_score'] = df['review_score'].astype(int)

df = df.drop(columns = ['review_comment_title', 'review_comment_message'])

print(f'Columns: {df.columns.tolist()}')
df.head()

Columns: ['order_id', 'customer_unique_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'customer_lat', 'customer_lng', 'shipping_limit_date', 'num_items', 'total_price', 'total_freight', 'seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'seller_lat', 'seller_lng', 'product_category_name', 'product_category_name_english', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_name_length', 'product_desc_length', 'num_payment_methods', 'payment_installments', 'payment_value', 'review_score', 'review_creation_date', 'review_answer_timestamp', 'paid_boleto', 'paid_credit_card', 'paid_debit_card', 'paid_voucher']


,order_id,customer_unique_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_zip_code_prefix,customer_city,...,num_payment_methods,payment_installments,payment_value,review_score,review_creation_date,review_answer_timestamp,paid_boleto,paid_credit_card,paid_debit_card,paid_voucher
0,00010242fe8c5a6d1ba2dd792cb16214,871766c5855e863f6eccc05f988b23cb,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,28013,campos dos goytacazes,...,1,2,72.19,5,2017-09-21,2017-09-22 10:57:03,False,True,False,False
1,00018f77f2f0320c557190d7a144bdd3,eb28e67c4c0b83846050ddfb8a35d051,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,15775,santa fe do sul,...,1,3,259.83,4,2017-05-13,2017-05-15 11:34:13,False,True,False,False
2,000229ec398224ef6ca0657da4fc703e,3818d81c6709e39d06b2738a8d3a2474,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,35661,para de minas,...,1,5,216.87,5,2018-01-23,2018-01-23 16:06:31,False,True,False,False
3,00024acbcdf0a6daa1e931b038114c75,af861d436cfc08b2c2ddefd0ba074622,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,12952,atibaia,...,1,2,25.78,4,2018-08-15,2018-08-15 16:39:01,False,True,False,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,64b576fb70d441e8f1b2d7d446e483c5,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,13226,varzea paulista,...,1,3,218.04,5,2017-03-02,2017-03-03 10:54:59,False,True,False,False


### 2. Safe feature engineering — delivery, pricing, product, payment, calendar

Pure arithmetic on columns within the same row.
Safe to call identically on train and test with zero risk of leakage.

In [5]:
def build_safe_features(df):
    df = df.copy()

    _date_cols = [
        'order_purchase_timestamp', 'order_approved_at',
        'order_delivered_carrier_date', 'order_delivered_customer_date',
        'order_estimated_delivery_date', 'shipping_limit_date',
        'review_creation_date', 'review_answer_timestamp',
    ]
    for _col in _date_cols:
        if _col in df.columns and not pd.api.types.is_datetime64_any_dtype(df[_col]):
            df[_col] = pd.to_datetime(df[_col], errors='coerce')

    price_col   = 'total_price'   if 'total_price'   in df.columns else 'price'
    freight_col = 'total_freight' if 'total_freight' in df.columns else 'freight_value'
    desc_col    = 'product_desc_length'  if 'product_desc_length'  in df.columns else 'product_description_lenght'
    name_col    = 'product_name_length'  if 'product_name_length'  in df.columns else 'product_name_lenght'
    items_col   = 'num_items' if 'num_items' in df.columns else 'order_item_id'

    # Haversine distance seller → customer (great-circle km)
    def haversine(lat1, lon1, lat2, lon2):
        R = 6371
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
        return R * 2 * np.arcsin(np.sqrt(a))

    df['geo_distance_km'] = haversine(
        df['seller_lat'], df['seller_lng'],
        df['customer_lat'], df['customer_lng']
    )

    # Delivery timing
    df['estimated_wait_days'] = (
        (df['order_estimated_delivery_date'] - df['order_purchase_timestamp'])
        .dt.total_seconds() / 86_400
    ).clip(lower=0.01)

    df['delivery_delay_days'] = (
        (df['order_delivered_customer_date'] - df['order_estimated_delivery_date'])
        .dt.total_seconds() / 86_400
    )

    df['delivery_delay_ratio'] = df['delivery_delay_days'] / df['estimated_wait_days']

    df['end_to_end_delivery_days'] = (
        (df['order_delivered_customer_date'] - df['order_purchase_timestamp'])
        .dt.total_seconds() / 86_400
    )

    # Actual / estimated — ratio >1 means slower than promised (different from delay_ratio)
    df['estimated_vs_actual_ratio'] = (
        df['end_to_end_delivery_days'] / df['estimated_wait_days'].clip(lower=0.01)
    )

    df['processing_time_hrs'] = (
        (df['order_approved_at'] - df['order_purchase_timestamp'])
        .dt.total_seconds() / 3600
    )

    df['carrier_pickup_lag'] = (
        (df['order_delivered_carrier_date'] - df['order_approved_at'])
        .dt.total_seconds() / 86_400
    )

    # Seller dispatch: approval → carrier handoff (seller's own responsibility window)
    df['approval_to_carrier_days'] = (
        (df['order_delivered_carrier_date'] - df['order_approved_at'])
        .dt.total_seconds() / 86_400
    )

    df['days_to_ship_limit'] = (
        (df['shipping_limit_date'] - df['order_purchase_timestamp'])
        .dt.total_seconds() / 86_400
    )

    df['days_before_ship_limit'] = (
        (df['shipping_limit_date'] - df['order_delivered_carrier_date'])
        .dt.total_seconds() / 86_400
    )

    df['days_to_review'] = (
        (df['review_creation_date'] - df['order_delivered_customer_date'])
        .dt.total_seconds() / 86_400
    )

    # Review speed bucket: 0=fast(<2d), 1=medium(2-7d), 2=slow(>7d)
    df['review_speed_bucket'] = pd.cut(
        df['days_to_review'],
        bins=[-np.inf, 2, 7, np.inf],
        labels=[0, 1, 2]
    ).astype(float)

    # Freight & pricing
    df['freight_to_price'] = np.where(df[price_col] > 0,
        df[freight_col] / df[price_col], np.nan)

    df['freight_fraction'] = np.where(
        df[price_col] + df[freight_col] > 0,
        df[freight_col] / (df[price_col] + df[freight_col]), np.nan)

    df['price_per_item']   = df[price_col]   / df[items_col].clip(lower=1)
    df['freight_per_item'] = df[freight_col] / df[items_col].clip(lower=1)

    # Payment features
    df['value_per_installment'] = np.where(
        df['payment_installments'] > 0,
        df['payment_value'] / df['payment_installments'], np.nan)

    methods_col = 'num_payment_methods' if 'num_payment_methods' in df.columns else 'payment_sequential'
    df['multi_payment']         = (df[methods_col] > 1).astype(int)
    df['is_single_installment'] = (df['payment_installments'] == 1).astype(int)
    df['high_installments']     = (df['payment_installments'] >= 6).astype(int)

    df['freight_installment_ratio'] = np.where(
        df['payment_installments'] > 0,
        df[freight_col] / df['payment_installments'], np.nan)

    # Product features
    df['has_description']    = (df[desc_col].fillna(0) > 0).astype(int)
    df['has_photos']         = (df['product_photos_qty'].fillna(0) > 0).astype(int)
    df['photo_qty']          = df['product_photos_qty'].fillna(0)
    df['total_listing_length'] = df[name_col].fillna(0) + df[desc_col].fillna(0)

    df['product_volume_cm3'] = (
        df['product_length_cm'] * df['product_height_cm'] * df['product_width_cm']
    )

    # Density: heavy-for-size → expensive to ship, affects carrier handling
    df['weight_to_volume_ratio'] = np.where(
        df['product_volume_cm3'] > 0,
        df['product_weight_g'] / df['product_volume_cm3'], np.nan)

    # Listing effort: many photos relative to description length
    df['photos_per_desc_char'] = np.where(
        df[desc_col].fillna(0) > 0,
        df['product_photos_qty'].fillna(0) / df[desc_col].clip(lower=1), np.nan)

    df['is_heavy_product'] = (df['product_weight_g'] > 5_000).astype(int)
    df['is_bulky_product'] = (df['product_volume_cm3'] > 40_000).astype(int)

    # Calendar / behavioural 
    df['purchase_hour']    = df['order_purchase_timestamp'].dt.hour
    df['purchase_dow']     = df['order_purchase_timestamp'].dt.dayofweek
    df['purchase_month']   = df['order_purchase_timestamp'].dt.month
    df['purchase_quarter'] = df['order_purchase_timestamp'].dt.quarter
    df['is_weekend']       = (df['purchase_dow'] >= 5).astype(int)
    df['is_night']         = (df['purchase_hour'].between(22, 5)).astype(int)

    # Geographic
    df['same_state'] = (df['customer_state'] == df['seller_state']).astype(int)

    # Region dummies (deterministic map — safe before split)
    region_map = {
        'SP':'southeast','RJ':'southeast','MG':'southeast','ES':'southeast',
        'RS':'south',    'SC':'south',    'PR':'south',
        'BA':'northeast','PE':'northeast','CE':'northeast','MA':'northeast',
        'PB':'northeast','RN':'northeast','AL':'northeast','SE':'northeast','PI':'northeast',
        'GO':'centerwest','MT':'centerwest','MS':'centerwest','DF':'centerwest',
        'AM':'north','PA':'north','RO':'north','AC':'north',
        'RR':'north','AP':'north','TO':'north',
    }
    df['customer_region'] = df['customer_state'].map(region_map)
    df['seller_region']   = df['seller_state'].map(region_map)

    # Use pd.get_dummies then drop_first to avoid the dummy-variable trap
    cust_dummies = pd.get_dummies(df['customer_region'], prefix='cust_reg', drop_first=True).astype(int)
    sell_dummies = pd.get_dummies(df['seller_region'],   prefix='sell_reg', drop_first=True).astype(int)
    df = pd.concat([df, cust_dummies, sell_dummies], axis=1)
    df = df.drop(columns=['customer_region', 'seller_region'])

    return df

df = build_safe_features(df)

df.head()

,order_id,customer_unique_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_zip_code_prefix,customer_city,...,is_night,same_state,cust_reg_north,cust_reg_northeast,cust_reg_south,cust_reg_southeast,sell_reg_north,sell_reg_northeast,sell_reg_south,sell_reg_southeast
0,00010242fe8c5a6d1ba2dd792cb16214,871766c5855e863f6eccc05f988b23cb,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,28013,campos dos goytacazes,...,0,0,0,0,0,1,0,0,0,1
1,00018f77f2f0320c557190d7a144bdd3,eb28e67c4c0b83846050ddfb8a35d051,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,15775,santa fe do sul,...,0,1,0,0,0,1,0,0,0,1
2,000229ec398224ef6ca0657da4fc703e,3818d81c6709e39d06b2738a8d3a2474,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,35661,para de minas,...,0,1,0,0,0,1,0,0,0,1
3,00024acbcdf0a6daa1e931b038114c75,af861d436cfc08b2c2ddefd0ba074622,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,12952,atibaia,...,0,1,0,0,0,1,0,0,0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,64b576fb70d441e8f1b2d7d446e483c5,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,13226,varzea paulista,...,0,0,0,0,0,1,0,0,1,0


### 3. Interaction features (safe)

Arithmetic combinations of safe features — no cross-row aggregation.

In [ ]:
# Late AND expensive shipping — double-bad signal
df['delay_x_freight_ratio'] = df['delivery_delay_days'] * df['freight_to_price']

# Complex multi-item orders put more pressure on processing
items_col = 'num_items' if 'num_items' in df.columns else 'order_item_id'
df['processing_x_items'] = df['processing_time_hrs'] * df[items_col]

# High installments on expensive orders = credit burden proxy
df['installment_burden'] = df['payment_installments'] * df['payment_value']

# Delay relative to how long the customer expected to wait
df['delay_per_estimated_day'] = (
    df['delivery_delay_days'] / df['estimated_wait_days'].clip(lower=1)
)
# Heavy parcel + long distance = highest shipping risk composite
df['weight_x_distance'] = df['product_weight_g'] * df['geo_distance_km']

df.head()

df.to_csv('../Data/pre_split_features_nb.csv', index=False)

### 4. Train / test split

Sklearn train_test_split, stratified on review_score.
The **test set is locked** from this point — no statistics from it touch feature engineering.

In [7]:
train_fe, test_fe = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['review_score']
)

train_fe = train_fe.reset_index(drop=True)
test_fe  = test_fe.reset_index(drop=True)


### 5. Fold-aware feature engineering

These aggregate across rows and carry leakage risk if computed on the full dataset before splitting.

The function always takes two arguments: train_df (the fold whose statistics are computed) and apply_df (the fold that receives the mapped values). The two are often different — you fit on training, then apply to validation or test.

Re-call build_foldaware_features(fold_train, fold_val) inside every CV fold.

Calculate the training ones and then map both set of values to train and test sets

In [8]:
def build_foldaware_features(train_df, apply_df):

    apply_df = apply_df.copy()

    # Global fallbacks computed from train only
    global_mean_freight  = train_df['freight_to_price'].mean()
    global_install       = train_df['payment_installments'].mean()

    # Customer zip-code installment norm and deviation
    # Captures whether this customer is borrowing more than is normal for their area
    zip_install = train_df.groupby('customer_zip_code_prefix')['payment_installments'].mean()
    apply_df['fbi_zip_install_mean'] = (
        apply_df['customer_zip_code_prefix'].map(zip_install).fillna(global_install)
    )
    apply_df['fbi_installment_deviation'] = (
        apply_df['payment_installments'] - apply_df['fbi_zip_install_mean']
    )

    # Customer zip freight norm
    # Captures whether this order's shipping cost is high relative to others in the same area
    zip_freight = train_df.groupby('customer_zip_code_prefix')['freight_to_price'].mean()
    apply_df['zip_avg_freight'] = (
        apply_df['customer_zip_code_prefix'].map(zip_freight).fillna(global_mean_freight)
    )

    # Seller statistics (by seller_id)
    # A seller's historical delay, on-time rate, and volume are strong signals
    # None of these use review_score
    if 'seller_id' in train_df.columns:
        seller_stats = train_df.groupby('seller_id').agg(
            seller_avg_delay        =('delivery_delay_days', 'mean'),
            seller_delay_std        =('delivery_delay_days', 'std'),
            seller_on_time_rate     =('delivery_delay_days', lambda x: (x <= 0).mean()),
            seller_order_volume     =('order_id', 'count'),
            seller_avg_processing_hrs=('processing_time_hrs', 'mean'),
        )
        global_seller_delay       = train_df['delivery_delay_days'].mean()
        global_seller_on_time     = (train_df['delivery_delay_days'] <= 0).mean()
        global_seller_volume      = seller_stats['seller_order_volume'].mean()
        global_processing         = train_df['processing_time_hrs'].mean()

        apply_df['seller_avg_delay']         = apply_df['seller_id'].map(seller_stats['seller_avg_delay']).fillna(global_seller_delay)
        apply_df['seller_delay_std']         = apply_df['seller_id'].map(seller_stats['seller_delay_std']).fillna(0)
        apply_df['seller_on_time_rate']      = apply_df['seller_id'].map(seller_stats['seller_on_time_rate']).fillna(global_seller_on_time)
        apply_df['seller_order_volume']      = apply_df['seller_id'].map(seller_stats['seller_order_volume']).fillna(1)
        apply_df['seller_avg_processing_hrs']= apply_df['seller_id'].map(seller_stats['seller_avg_processing_hrs']).fillna(global_processing)

        # How much worse/better is this specific order's freight vs the seller's average
        seller_avg_freight = train_df.groupby('seller_id')['freight_to_price'].mean()
        apply_df['freight_vs_seller_avg'] = (
            apply_df['freight_to_price']
            - apply_df['seller_id'].map(seller_avg_freight).fillna(global_mean_freight)
        )

        # Combined signal: far away from an unreliable seller — highest delivery risk
        apply_df['distance_x_seller_reliability'] = (
            apply_df['geo_distance_km'] * (1 - apply_df['seller_on_time_rate'])
        )

    # Category statistics (by product_category_name)
    # Some categories structurally have higher freight, longer wait, heavier items
    cat_col = 'product_category_name'
    cat_stats = train_df.groupby(cat_col).agg(
        category_avg_price  =(('total_price'   if 'total_price'   in train_df.columns else 'price'),   'mean'),
        category_avg_freight=(('total_freight' if 'total_freight' in train_df.columns else 'freight_value'), 'mean'),
        category_avg_delay  =('delivery_delay_days', 'mean'),
        category_order_count=('order_id', 'count'),
    )
    price_col   = 'total_price'   if 'total_price'   in train_df.columns else 'price'
    freight_col = 'total_freight' if 'total_freight' in train_df.columns else 'freight_value'

    apply_df['category_avg_price']   = apply_df[cat_col].map(cat_stats['category_avg_price']).fillna(train_df[price_col].mean())
    apply_df['category_avg_freight'] = apply_df[cat_col].map(cat_stats['category_avg_freight']).fillna(train_df[freight_col].mean())
    apply_df['category_avg_delay']   = apply_df[cat_col].map(cat_stats['category_avg_delay']).fillna(train_df['delivery_delay_days'].mean())
    apply_df['category_order_count'] = apply_df[cat_col].map(cat_stats['category_order_count']).fillna(1)

    # Customer statistics (by customer_unique_id)
    # Repeat customers have calibrated expectations; their order history provides context
    if 'customer_unique_id' in train_df.columns:
        cust_stats = train_df.groupby('customer_unique_id').agg(
            customer_order_count      =('order_id', 'nunique'),
            customer_avg_payment      =(('payment_value'), 'mean'),
            customer_avg_installments =(('payment_installments'), 'mean'),
            customer_avg_freight_ratio=('freight_to_price', 'mean'),
        )
        apply_df['customer_order_count']       = apply_df['customer_unique_id'].map(cust_stats['customer_order_count']).fillna(1)
        apply_df['customer_avg_payment']       = apply_df['customer_unique_id'].map(cust_stats['customer_avg_payment']).fillna(train_df['payment_value'].mean())
        apply_df['customer_avg_installments']  = apply_df['customer_unique_id'].map(cust_stats['customer_avg_installments']).fillna(global_install)
        apply_df['customer_avg_freight_ratio'] = apply_df['customer_unique_id'].map(cust_stats['customer_avg_freight_ratio']).fillna(global_mean_freight)
        apply_df['customer_is_repeat']         = (apply_df['customer_order_count'] > 1).astype(int)

    foldaware_cols = [
        'fbi_zip_install_mean', 'fbi_installment_deviation', 'zip_avg_freight',
        'seller_avg_delay', 'seller_delay_std', 'seller_on_time_rate',
        'seller_order_volume', 'seller_avg_processing_hrs',
        'freight_vs_seller_avg', 'distance_x_seller_reliability',
        'category_avg_price', 'category_avg_freight', 'category_avg_delay', 'category_order_count',
        'customer_order_count', 'customer_avg_payment', 'customer_avg_installments',
        'customer_avg_freight_ratio', 'customer_is_repeat'
        ]
    return apply_df, foldaware_cols

train_fe, foldaware_cols = build_foldaware_features(train_fe, train_fe)
test_fe,  _              = build_foldaware_features(train_fe, test_fe)


In [9]:
train_fe.head()

,order_id,customer_unique_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_zip_code_prefix,customer_city,...,distance_x_seller_reliability,category_avg_price,category_avg_freight,category_avg_delay,category_order_count,customer_order_count,customer_avg_payment,customer_avg_installments,customer_avg_freight_ratio,customer_is_repeat
0,1daaf8be4f0713e6fd0b12a974e29110,6830cf7a24de7852d3a41a26dfa6e209,delivered,2018-05-09 10:57:58,2018-05-10 10:56:43,2018-05-10 14:23:00,2018-05-14 21:24:43,2018-05-24,4349,sao paulo,...,0.000000,86.166032,18.465864,-12.592652,1424,1,81.24,2.0,0.232777,0
1,038eb192a216fe0c545a8ff3b1b96885,b2b779eb0dedff7f9eef0770287a9979,delivered,2017-12-30 22:07:13,2017-12-30 22:19:19,2018-01-03 22:14:41,2018-01-05 18:23:09,2018-01-24,15015,sao jose do rio preto,...,0.000000,104.732323,21.304001,-11.277554,1752,1,51.75,5.0,0.296992,0
2,c5dc5eee680080a268100b420f8e960a,f0dc4035c184f9d29bf0d74c8a57aa9a,delivered,2018-05-31 15:33:14,2018-05-31 15:51:47,2018-06-04 14:34:00,2018-06-05 22:08:23,2018-06-26,4445,sao paulo,...,1.224218,148.783903,21.527722,-11.270098,6884,1,336.51,3.0,0.246333,0
3,9ab2cf8063bac26a51227f2ebe58a554,7abf56cce374378793ed6118bb4eb9d9,delivered,2017-05-11 18:01:55,2017-05-11 18:15:22,2017-05-12 14:47:58,2017-05-24 09:39:34,2017-06-06,89770,seara,...,16.789259,143.886962,31.102089,-12.132749,790,1,114.80,10.0,0.209694,0
4,488470d4668abcf0aa20df845d584dc2,b0fbf3dd205360c209e99fb44a035ba5,delivered,2017-08-23 10:12:24,2017-08-23 10:32:13,2017-08-23 18:38:23,2017-08-28 18:21:32,2017-09-18,7414,aruja,...,16.755480,133.582601,22.814857,-11.283945,5975,1,80.20,1.0,0.233846,0


### 6. Composite motif indices

Each index z-scores its sub-features using train statistics only, then averages.
Z scores cus on diff scales...


- Frustration Index: relative delivery delay + freight burden
- Financial Burden Index: per-installment cost + deviation from local norm
- Consistency Index: order complexity + impulse-buying timing signal

In [10]:
def zscore(series, mu=None, sigma=None):
    if mu is None:    
        mu = series.mean()

    if sigma is None: 
        sigma = series.std()
    
    # case where all is identical (just return all zeros...)
    if sigma == 0:
        return pd.Series(np.zeros(len(series)), index=series.index)
    
    return (series - mu) / sigma


def build_motif_indices(df, z_stats=None, fit=True):

    d = df.copy()
    if z_stats is None:
        z_stats = {}

    def _z(col):

        if fit:
            mu = d[col].mean()
            sigma = d[col].std()
            z_stats[col] = (mu, sigma)

        else:
            mu, sigma = z_stats.get(col, (0, 1))

        return zscore(d[col], mu, sigma)

    # Frustration Index: late + expensive freight
    d['frustration_index'] = (
        _z('delivery_delay_ratio').fillna(0) +
        _z('freight_to_price').fillna(0)
    ) / 2

    # Financial Burden Index: installment cost + how unusual for this customer's area
    d['financial_burden_index'] = (
        _z('value_per_installment').fillna(0) +
        _z('fbi_installment_deviation').fillna(0)
    ) / 2

    # Consistency Index: order size + price spread + impulse purchase timing
    if 'num_items' in d.columns:
        items_col = 'num_items'

    else:
        'order_item_id'

    d['consistency_index'] = (
        _z(items_col).fillna(0) +
        _z('price_per_item').fillna(0) +
        _z('is_night').fillna(0)
    ) / 3

    return d, z_stats


train_fe, z_stats = build_motif_indices(train_fe, fit=True)
test_fe,  _ = build_motif_indices(test_fe, z_stats=z_stats, fit=False)


In [11]:
train_fe.head()
train_fe.columns.to_list()

['order_id',
 'customer_unique_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'customer_lat',
 'customer_lng',
 'shipping_limit_date',
 'num_items',
 'total_price',
 'total_freight',
 'seller_id',
 'seller_zip_code_prefix',
 'seller_city',
 'seller_state',
 'seller_lat',
 'seller_lng',
 'product_category_name',
 'product_category_name_english',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm',
 'product_name_length',
 'product_desc_length',
 'num_payment_methods',
 'payment_installments',
 'payment_value',
 'review_score',
 'review_creation_date',
 'review_answer_timestamp',
 'paid_boleto',
 'paid_credit_card',
 'paid_debit_card',
 'paid_voucher',
 'geo_distance_km',
 'estimated_wait_days',
 'delivery_delay_days',
 'delivery_delay_ratio'

### 7. Assemble final feature matrix

In [12]:
drop_cols = [
    # IDs
    'order_id', 'seller_id', 'customer_unique_id',
    # Raw datetimes -> which we have extracted from
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date', 'shipping_limit_date',
    'review_creation_date', 'review_answer_timestamp',
    # Categoricals now encoded
    'product_category_name', 'product_category_name_english',
    'customer_city', 'seller_city',
    'customer_state', 'seller_state',
    'order_status',
    # Duplicates
    'carrier_pickup_lag',
    'photo_qty',
]

TARGET_COL = 'review_score'

def prepare_X(df, drop_cols, target_col):
    d = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
    d = d.drop(columns=[target_col], errors='ignore')

    # Drop any surviving non-numeric cols (object, category, datetime)
    bad = d.select_dtypes(include=['object','category','datetime64[ns]']).columns.tolist()
    if bad:
        d = d.drop(columns=bad)

    # Median-impute remaining NaNs
    for col in d.columns:
        if d[col].isna().any():
            d[col] = d[col].fillna(d[col].median())

    return d


X_train_df = prepare_X(train_fe, drop_cols, TARGET_COL)
X_test_df  = prepare_X(test_fe,  drop_cols, TARGET_COL)

# Align columns (test must match train exactly)
for col in set(X_train_df.columns) - set(X_test_df.columns):
    X_test_df[col] = 0
X_test_df = X_test_df[X_train_df.columns]

y_train = train_fe[TARGET_COL].values.astype(int)
y_test  = test_fe[TARGET_COL].values.astype(int)



In [13]:
X_train_df.head()

,customer_zip_code_prefix,customer_lat,customer_lng,num_items,total_price,total_freight,seller_zip_code_prefix,seller_lat,seller_lng,product_photos_qty,...,category_avg_delay,category_order_count,customer_order_count,customer_avg_payment,customer_avg_installments,customer_avg_freight_ratio,customer_is_repeat,frustration_index,financial_burden_index,consistency_index
0,4349,-23.648721,-46.646756,1,65.9,15.34,31980,-19.854615,-43.914740,4,...,-12.592652,1424,1,81.24,2.0,0.232777,0,-0.338350,-0.267535,-0.188550
1,15015,-20.819121,-49.379028,1,39.9,11.85,14801,-21.790783,-48.185502,1,...,-11.277554,1752,1,51.75,5.0,0.296992,0,-0.410654,0.365115,-0.228428
2,4445,-23.683200,-46.685266,3,90.0,22.17,2028,-23.508703,-46.618337,2,...,-11.270098,6884,1,336.51,3.0,0.246333,0,-0.536949,0.211049,0.998145
3,89770,-27.160802,-52.315400,1,94.9,19.90,37564,-22.262584,-46.171124,1,...,-12.132749,790,1,114.80,10.0,0.209694,0,-0.205788,1.191754,-0.144071
4,7414,-23.406307,-46.364838,1,65.0,15.20,89160,-27.214587,-49.637725,1,...,-11.283945,5975,1,80.20,1.0,0.233846,0,-0.555745,-0.024745,-0.189930


### 9. Save the df

In [14]:
import os
os.makedirs('../Data', exist_ok=True)

X_train = X_train_df.values.astype(np.float64)
X_test  = X_test_df.values.astype(np.float64)

np.save('../Data/X_train.npy', X_train)
np.save('../Data/X_test.npy',  X_test)
np.save('../Data/y_train.npy', y_train)
np.save('../Data/y_test.npy',  y_test)

pd.Series(X_train_df.columns.tolist()).to_csv(
    '../Data/feature_names.csv', index=False, header=False)
train_fe.to_csv('../Data/train_features.csv', index=False)
test_fe.to_csv( '../Data/test_features.csv',  index=False)
